# 06 — Error Handling & Debugging

Proper error handling separates junior developers from senior ones. Interviewers love to probe how you think about failure modes.

---

## Table of Contents
1. Error Types in Node.js
2. Operational vs Programmer Errors
3. Error Handling Patterns
4. Uncaught Exceptions & Unhandled Rejections
5. Custom Error Classes
6. Debugging Techniques
7. Logging Best Practices
8. Interview Questions

---
## 1. Error Types in Node.js

| Error Type | Description | Example |
|-----------|-----------|--------|
| `Error` | Generic error | `new Error('msg')` |
| `TypeError` | Wrong type | `null.property` |
| `RangeError` | Value out of range | `new Array(-1)` |
| `ReferenceError` | Undeclared variable | `console.log(x)` |
| `SyntaxError` | Invalid code | `JSON.parse('{bad}')` |
| `SystemError` | OS-level error | `ENOENT`, `ECONNREFUSED` |

In [ ]:
// Error object anatomy
const err = new Error('Something went wrong');

console.log('name:', err.name);           // 'Error'
console.log('message:', err.message);     // 'Something went wrong'
console.log('stack:', err.stack?.split('\n')[0]); // First line of stack trace

// System errors (from Node.js core)
const fs = require('fs');
try {
    fs.readFileSync('/nonexistent/file.txt');
} catch (err) {
    console.log('\nSystem error:');
    console.log('code:', err.code);       // 'ENOENT'
    console.log('syscall:', err.syscall); // 'open'
    console.log('path:', err.path);       // '/nonexistent/file.txt'
}

---
## 2. Operational vs Programmer Errors

This distinction is **critical** for interviews.

### Operational Errors (expected, recoverable):
- Database connection timeout
- User sends invalid input
- File not found
- API rate limit exceeded
- Network request failed

**How to handle:** Respond gracefully. Return error response to client. Retry. Log and alert.

### Programmer Errors (bugs, NOT recoverable):
- Reading property of `undefined`
- Passing wrong types to functions
- Failing to handle a rejected Promise
- Off-by-one errors

**How to handle:** Fix the bug in code. In production, crash and restart (using PM2, Docker, etc.) because the process is in an unknown state.

> **Interview Key Insight:** "Operational errors should be anticipated and handled gracefully. Programmer errors should crash the process because continuing with corrupted state is worse than restarting."

In [ ]:
// Operational error — handle gracefully
async function fetchUserFromDB(id) {
    // Simulating a DB call
    if (id < 0) {
        const err = new Error('Invalid user ID');
        err.statusCode = 400;
        err.isOperational = true;
        throw err;
    }
    if (id > 1000) {
        const err = new Error('User not found');
        err.statusCode = 404;
        err.isOperational = true;
        throw err;
    }
    return { id, name: 'Alice' };
}

// Programmer error — crash is appropriate
// function processUser(user) {
//     return user.name.toUpperCase(); // If user is undefined → TypeError (programmer bug)
// }

async function demo() {
    try {
        await fetchUserFromDB(9999);
    } catch (err) {
        if (err.isOperational) {
            console.log(`Operational error (${err.statusCode}): ${err.message}`);
        } else {
            console.error('PROGRAMMER ERROR — should not happen:', err);
            // process.exit(1); // In production
        }
    }
}
demo();

---
## 3. Error Handling Patterns

In [ ]:
// Pattern 1: try/catch for synchronous code
function parseJSON(str) {
    try {
        return { data: JSON.parse(str), error: null };
    } catch (err) {
        return { data: null, error: err.message };
    }
}

console.log(parseJSON('{"valid": true}'));  // { data: { valid: true }, error: null }
console.log(parseJSON('not json'));          // { data: null, error: '...' }

In [ ]:
// Pattern 2: Error-first callbacks
const fs = require('fs');

fs.readFile('/nonexistent', 'utf8', (err, data) => {
    if (err) {
        console.log('Error-first callback:', err.code); // ENOENT
        return; // Always return after error!
    }
    console.log(data);
});

In [ ]:
// Pattern 3: Promise .catch()
const fsPromises = require('fs').promises;

fsPromises.readFile('/nonexistent', 'utf8')
    .then(data => console.log(data))
    .catch(err => console.log('Promise catch:', err.code));

In [ ]:
// Pattern 4: async/await with try/catch
async function readFileSafe(path) {
    try {
        const data = await require('fs').promises.readFile(path, 'utf8');
        return data;
    } catch (err) {
        if (err.code === 'ENOENT') {
            console.log('File not found — returning default');
            return 'default content';
        }
        throw err; // Re-throw unknown errors
    }
}

readFileSafe('/nonexistent').then(data => console.log('Got:', data));

In [ ]:
// Pattern 5: Go-style error handling (popular in Node.js)
async function to(promise) {
    try {
        const data = await promise;
        return [null, data];
    } catch (err) {
        return [err, null];
    }
}

async function demo() {
    const [err, data] = await to(require('fs').promises.readFile('/nonexistent'));
    if (err) {
        console.log('Go-style error:', err.code);
        return;
    }
    console.log(data);
}
demo();

---
## 4. Uncaught Exceptions & Unhandled Rejections

These are your **last line of defense** — they catch errors that no other handler caught.

In [ ]:
// Uncaught Exception — synchronous error no one caught
process.on('uncaughtException', (err) => {
    console.error('UNCAUGHT EXCEPTION:', err.message);
    // Log the error, close connections, then EXIT
    // process.exit(1); // DO NOT continue running!
});

// Unhandled Promise Rejection — async error no one caught
process.on('unhandledRejection', (reason, promise) => {
    console.error('UNHANDLED REJECTION:', reason);
    // In Node.js 15+, this crashes the process by default
});

console.log('Global error handlers registered');
console.log('IMPORTANT: Always exit after uncaughtException — the process state is corrupted');

### Graceful Shutdown Pattern:
```javascript
process.on('uncaughtException', (err) => {
    logger.error('Uncaught Exception', err);
    gracefulShutdown(1);
});

process.on('unhandledRejection', (reason) => {
    logger.error('Unhandled Rejection', reason);
    gracefulShutdown(1);
});

process.on('SIGTERM', () => gracefulShutdown(0));
process.on('SIGINT', () => gracefulShutdown(0));

async function gracefulShutdown(code) {
    console.log('Shutting down gracefully...');
    await server.close();         // Stop accepting new connections
    await db.disconnect();        // Close DB connections
    await cache.quit();           // Close Redis, etc.
    process.exit(code);
}
```

---
## 5. Custom Error Classes

In [ ]:
// Production-quality custom error hierarchy

class AppError extends Error {
    constructor(message, statusCode) {
        super(message);
        this.statusCode = statusCode;
        this.isOperational = true;
        Error.captureStackTrace(this, this.constructor);
    }
}

class NotFoundError extends AppError {
    constructor(resource = 'Resource') {
        super(`${resource} not found`, 404);
    }
}

class ValidationError extends AppError {
    constructor(message) {
        super(message, 400);
    }
}

class UnauthorizedError extends AppError {
    constructor(message = 'Unauthorized') {
        super(message, 401);
    }
}

class ForbiddenError extends AppError {
    constructor(message = 'Forbidden') {
        super(message, 403);
    }
}

// Usage
try {
    throw new NotFoundError('User');
} catch (err) {
    console.log(`${err.statusCode}: ${err.message}`);
    console.log('Is operational:', err.isOperational);
    console.log('Is AppError:', err instanceof AppError);
    console.log('Is NotFoundError:', err instanceof NotFoundError);
}

---
## 6. Debugging Techniques

### Method 1: `console` methods
```javascript
console.log()        // General output
console.error()      // Error output (stderr)
console.warn()       // Warning
console.table()      // Display arrays/objects as table
console.time()       // Start timer
console.timeEnd()    // End timer
console.trace()      // Print stack trace
console.dir(obj, { depth: null }) // Deep object inspection
```

### Method 2: Built-in debugger
```bash
node --inspect app.js           # Start with debugger
node --inspect-brk app.js       # Break on first line
# Then open chrome://inspect in Chrome
```

### Method 3: VS Code debugger
- Add breakpoints in the gutter
- Press F5 or use "Run and Debug"
- Inspect variables, call stack, watch expressions

### Method 4: `debug` npm package
```javascript
const debug = require('debug')('app:server');
debug('Server starting on port %d', 3000);
// Enable with: DEBUG=app:* node app.js
```

In [ ]:
// Useful debugging techniques

// console.table for arrays of objects
const users = [
    { name: 'Alice', role: 'admin', active: true },
    { name: 'Bob', role: 'user', active: false },
    { name: 'Charlie', role: 'user', active: true },
];
console.table(users);

// Timing operations
console.time('operation');
let sum = 0;
for (let i = 0; i < 1000000; i++) sum += i;
console.timeEnd('operation');

// Stack trace
function a() { b(); }
function b() { c(); }
function c() { console.trace('Where am I?'); }
a();

---
## 7. Logging Best Practices

### Log levels (from most to least severe):
```
FATAL  → Application cannot continue
ERROR  → Something failed, but app continues
WARN   → Something unexpected, but not broken
INFO   → Key business events (user login, order placed)
DEBUG  → Detailed flow info (for development)
TRACE  → Very verbose (function entry/exit)
```

### Popular logging libraries:
- **Winston** — Most popular, flexible, multiple transports
- **Pino** — Fastest, JSON-based, great for production
- **Bunyan** — JSON-focused, good for structured logging

### What to log:
- Request ID (correlation ID) for tracing
- User ID (who triggered it)
- Timestamp
- Error message + stack trace
- Request method, URL, status code, duration

### What NOT to log:
- Passwords, tokens, API keys
- Personal data (unless compliant with GDPR/regulations)
- Entire request/response bodies in production

---
## 8. Interview Questions & Answers

### Q1: What's the difference between operational and programmer errors?
**A:** Operational errors are expected runtime problems (network timeouts, invalid input, file not found) — handle them gracefully. Programmer errors are bugs (TypeError, null reference) — the process is in an unknown state and should be restarted.

### Q2: How do you handle uncaught exceptions in production?
**A:** Listen for `uncaughtException` and `unhandledRejection` events on `process`. Log the error, close open connections gracefully, then exit. Use a process manager (PM2, Docker) to automatically restart. Never continue running after an uncaught exception.

### Q3: How do you implement centralized error handling in Express?
**A:** Create an error-handling middleware with 4 parameters `(err, req, res, next)` as the last middleware. Use custom error classes with statusCode. Create an `asyncHandler` wrapper to catch Promise rejections. Distinguish operational errors (send to client) from programmer errors (log and return 500).

### Q4: What is `Error.captureStackTrace`?
**A:** A V8-specific method that captures the current stack trace and assigns it to an error object. When used in custom error constructors with `Error.captureStackTrace(this, this.constructor)`, it excludes the constructor itself from the stack trace, making the trace cleaner.

### Q5: How do you debug a memory leak in Node.js?
**A:** Use `process.memoryUsage()` to monitor heap size. Use `--inspect` flag with Chrome DevTools to take heap snapshots. Compare snapshots to find objects that aren't being garbage collected. Common causes: global variables, closures holding references, event listeners not removed, growing caches without eviction.